# Parcel-state YOLO11n training
Train a separate detector whose only classes are `closed_box` and `open_box`.

In [ ]:
!nvidia-smi
!pip install -q ultralytics==8.4.129

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, yaml, torch

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
uploaded = files.upload()
archive_name = 'parcel_state_dataset.zip'
assert archive_name in uploaded, f'Upload {archive_name}; received {list(uploaded)}'
dataset_root = Path('/content/parcel_state_dataset')
if dataset_root.exists(): shutil.rmtree(dataset_root)
shutil.unpack_archive('/content/' + archive_name, dataset_root)
yaml_files = list(dataset_root.rglob('data.yaml'))
assert len(yaml_files) == 1, f'Expected one data.yaml, found {yaml_files}'
export_root = yaml_files[0].parent
print('Dataset root:', export_root)

In [ ]:
expected_names = ['closed_box', 'open_box']
source_config = yaml.safe_load((export_root/'data.yaml').read_text())
names = source_config['names']
if isinstance(names, dict): names = [names[i] if i in names else names[str(i)] for i in range(len(names))]
assert names == expected_names, f'Expected {expected_names}, received {names}'
counts = {}
for split in ('train', 'valid'):
    image_dir, label_dir = export_root/split/'images', export_root/split/'labels'
    assert image_dir.is_dir() and label_dir.is_dir(), f'Missing {split} images/labels'
    labels = list(label_dir.glob('*.txt'))
    class_counts = [0, 0]
    for label in labels:
        for line in label.read_text().splitlines():
            parts = line.split(); assert len(parts) >= 5 and len(parts[1:]) % 2 == 0, (label, line)
            class_id = int(parts[0]); assert class_id in (0, 1), (label, class_id)
            values = [float(v) for v in parts[1:]]
            assert all(0 <= v <= 1 for v in values)
            class_counts[class_id] += 1
    assert all(count > 0 for count in class_counts), f'{split} lacks a class: {class_counts}'
    counts[split] = class_counts
print('Box counts [closed_box, open_box]:', counts)
runtime_yaml = Path('/content/parcel_state_data.yaml')
runtime_yaml.write_text(yaml.safe_dump({'path': str(export_root), 'train': 'train/images', 'val': 'valid/images', 'nc': 2, 'names': expected_names}, sort_keys=False))
print(runtime_yaml.read_text())

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data=str(runtime_yaml), imgsz=640, epochs=100, batch=16, device=0, workers=2, pretrained=True, seed=42, deterministic=True, patience=20, project='/content/runs', name='PARCEL-STATE-001_yolo11n', exist_ok=False, plots=True, val=True, save=True)

In [ ]:
run_dir = Path('/content/runs/PARCEL-STATE-001_yolo11n')
best = run_dir/'weights'/'best.pt'; assert best.is_file()
best_model = YOLO(str(best))
assert list(best_model.names.values()) == expected_names, best_model.names
metrics = best_model.val(data=str(runtime_yaml), imgsz=640, device=0, plots=True)
print('precision:', metrics.box.mp, 'recall:', metrics.box.mr)
print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)

In [ ]:
checkpoint = Path('/content/parcel_state_best.pt')
shutil.copy2(best, checkpoint)
results_archive = shutil.make_archive('/content/parcel_state_training_results', 'zip', run_dir)
files.download(str(checkpoint))
files.download(results_archive)